# E-Commerce Inferential Statistics

**Dataset:** Online Retail Transactions (UCI)  
**Purpose:** Use statistical inference to draw conclusions, test hypotheses,
and quantify relationships in the data.  

This notebook builds directly on `EDA.ipynb` (data exploration) and
`descriptive_statistics.ipynb` (distributional properties).
Basic exploration and cleaning steps are not repeated here.

---

## 1. Setup & Imports

All inferential tests used in this notebook require the libraries below.
- **scipy.stats** — parametric and non-parametric tests, Q-Q plots
- **statsmodels** — OLS regression, ANOVA tables, Tukey HSD
- **pingouin** — effect sizes (Cohen's d, Cramér's V, eta-squared)


In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

try:
    import pingouin as pg
    HAS_PINGOUIN = True
except ImportError:
    HAS_PINGOUIN = False
    print('pingouin not installed — effect sizes will be computed manually.')

warnings.filterwarnings('ignore')
ALPHA = 0.05

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (11, 5),
                     'axes.titlesize': 14, 'axes.labelsize': 12})
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

print('Libraries loaded.  ALPHA =', ALPHA)

In [ ]:
_candidates = [
    os.path.join('..', 'data', 'raw', 'data.csv'),
    os.path.join('ecommerce_analysis', 'data', 'raw', 'data.csv'),
    os.path.join('..', '..', 'data', 'raw', 'data.csv'),
]
DATA_PATH = next((p for p in _candidates if os.path.exists(p)), None)
if DATA_PATH is None:
    raise FileNotFoundError('data.csv not found.')

df_raw = pd.read_csv(DATA_PATH, encoding='ISO-8859-1', low_memory=False)
print(f'Loaded {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns')

In [ ]:
df = df_raw.copy()

# Parse date
DATE_COL = None
for col in df.columns:
    if 'date' in col.lower() or 'time' in col.lower():
        try:
            df[col] = pd.to_datetime(df[col], infer_datetime_format=True)
            DATE_COL = col; break
        except Exception:
            pass

# Auto-detect key columns
num_raw      = df.select_dtypes(include='number').columns.tolist()
QTY_COL      = next((c for c in num_raw if 'qty' in c.lower() or 'quant' in c.lower()), None)
PRICE_COL    = next((c for c in num_raw if 'price' in c.lower() or 'unit' in c.lower()), None)
INVOICE_COL  = next((c for c in df.columns if 'invoice' in c.lower()), None)
CUSTOMER_COL = next((c for c in df.columns if 'customer' in c.lower() or 'cust' in c.lower()), None)
COUNTRY_COL  = next((c for c in df.columns if 'country' in c.lower()), None)
DESC_COL     = next((c for c in df.columns if 'desc' in c.lower()), None)

if QTY_COL and PRICE_COL:
    df['Revenue'] = df[QTY_COL] * df[PRICE_COL]

NUMERIC_COLS = df.select_dtypes(include='number').columns.tolist()
CAT_COLS     = df.select_dtypes(include=['object', 'category']).columns.tolist()

# Working subset: positive-revenue transactions only
mask = pd.Series(True, index=df.index)
if QTY_COL:   mask &= df[QTY_COL] > 0
if 'Revenue' in df.columns: mask &= df['Revenue'] > 0
df_clean = df[mask].copy()

print(f'Numeric cols : {NUMERIC_COLS}')
print(f'Cat cols     : {CAT_COLS}')
print(f'Clean rows   : {len(df_clean):,}')
display(df_clean.head(5))

---
## 2. Normality Testing

Choosing between **parametric** (e.g., t-test, ANOVA) and **non-parametric**
(e.g., Mann-Whitney U, Kruskal-Wallis) tests depends on whether the data is
approximately normally distributed.

- **Shapiro-Wilk** — most powerful for small samples (n ≤ 5000).
- **D'Agostino-Pearson** — preferred for larger samples; combines skewness and kurtosis.

Results of this section will automatically drive test selection in Sections 4–6.

In [ ]:
SAMPLE_N = 5000
norm_rows = []

for col in NUMERIC_COLS:
    s = df_clean[col].dropna()
    if len(s) < 8:
        continue
    if len(s) <= SAMPLE_N:
        test_name = 'Shapiro-Wilk'
        stat, p   = stats.shapiro(s.sample(min(len(s), SAMPLE_N), random_state=42))
    else:
        test_name = "D'Agostino-Pearson"
        stat, p   = stats.normaltest(s)

    norm_rows.append({
        'column':    col,
        'n':         len(s),
        'test':      test_name,
        'statistic': round(float(stat), 4),
        'p_value':   round(float(p), 6),
        'normal?':   'YES' if p > ALPHA else 'NO',
    })

normality_df = pd.DataFrame(norm_rows).set_index('column')
display(normality_df)

NORMAL_COLS = normality_df[normality_df['normal?'] == 'YES'].index.tolist()
print(f'\nNormal     : {NORMAL_COLS}')
print(f'Non-normal : {[c for c in NUMERIC_COLS if c not in NORMAL_COLS]}')

In [ ]:
plot_num = [c for c in NUMERIC_COLS if df_clean[c].nunique() > 5]
n_rows_q = (len(plot_num) + 1) // 2
fig, axes = plt.subplots(n_rows_q, 2, figsize=(13, 4 * n_rows_q))
axes = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for i, col in enumerate(plot_num):
    s = df_clean[col].dropna().sample(min(2000, len(df_clean[col].dropna())), random_state=42)
    stats.probplot(s, dist='norm', plot=axes[i])
    label = 'normal' if col in NORMAL_COLS else 'non-normal'
    axes[i].set_title(f'Q-Q Plot: {col}  ({label})')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Q-Q Plots vs Normal Distribution', fontsize=15, y=1.01)
plt.tight_layout()
plt.show()

---
## 3. Confidence Intervals

A **95% confidence interval (CI)** means: if we repeated this sampling process
100 times, approximately 95 of the resulting intervals would contain the true
population mean.

In e-commerce terms: *'We are 95% confident that the true average Revenue per
transaction lies between £X and £Y.'*  
This is more informative than a single point estimate because it communicates
**uncertainty** — narrower intervals mean more reliable estimates.

We use the t-distribution (which is robust at large n by the Central Limit Theorem).

In [ ]:
ci_rows = []
for col in NUMERIC_COLS:
    s = df_clean[col].dropna()
    if len(s) < 2:
        continue
    n, mean, se = len(s), s.mean(), stats.sem(s)
    lo, hi = stats.t.interval(0.95, df=n - 1, loc=mean, scale=se)
    ci_rows.append({
        'column': col, 'n': n,
        'mean': round(mean, 4),
        'CI_lower_95': round(lo, 4),
        'CI_upper_95': round(hi, 4),
        'margin_of_error': round(hi - mean, 4),
    })

ci_df = pd.DataFrame(ci_rows).set_index('column')
display(ci_df)

In [ ]:
biz_cols = [c for c in ['Revenue', QTY_COL, PRICE_COL] if c and c in ci_df.index]
if not biz_cols:
    biz_cols = ci_df.index[:3].tolist()

fig, axes = plt.subplots(1, len(biz_cols), figsize=(5 * len(biz_cols), 5))
if len(biz_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, biz_cols):
    row = ci_df.loc[col]
    ax.errorbar(x=0, y=row['mean'],
                yerr=row['margin_of_error'],
                fmt='o', color='steelblue', capsize=12, capthick=2.5,
                markersize=11, linewidth=2.5)
    ax.set_xlim(-0.5, 0.5)
    ax.set_xticks([])
    ax.set_title(f'{col}\n95% Confidence Interval')
    ax.set_ylabel(col)
    ax.annotate(f'Mean = {row["mean"]:.2f}',
                xy=(0.06, 0.93), xycoords='axes fraction', fontsize=10,
                color='steelblue', fontweight='bold')
    ax.annotate(f'95% CI: [{row["CI_lower_95"]:.2f},  {row["CI_upper_95"]:.2f}]',
                xy=(0.06, 0.85), xycoords='axes fraction', fontsize=9)

plt.suptitle('95% Confidence Intervals — Key Business Metrics', fontsize=14)
plt.tight_layout()
plt.show()

# Plain-language summary
for col in biz_cols:
    row = ci_df.loc[col]
    print(f'  {col}: We are 95% confident the true mean is between '
          f'{row["CI_lower_95"]:.2f} and {row["CI_upper_95"]:.2f}.')

---
## 4. Hypothesis Testing — Means (t-tests)

Every test in this notebook follows the same structure:
```
H0 (null)       →  what we assume by default
H1 (alternative) →  what we want to support
Test statistic + p-value
Plain-language conclusion at α = 0.05
```

- **One-sample t-test**: is the mean of a metric different from a known benchmark?
- **Two-sample t-test (Welch)**: do two groups have different means?
- **Mann-Whitney U**: non-parametric alternative when normality is rejected.
  Tests whether one group tends to have higher values than the other (rank-sum).

In [ ]:
# ── One-sample t-test ────────────────────────────────────────────────────────
TARGET_COL = 'Revenue' if 'Revenue' in df_clean.columns else NUMERIC_COLS[0]
BENCHMARK  = 20.0  # hypothetical industry benchmark — adjust as needed

sample = df_clean[TARGET_COL].dropna()
t_stat, p_val = stats.ttest_1samp(sample, popmean=BENCHMARK)

print('ONE-SAMPLE T-TEST')
print(f'  Column     : {TARGET_COL}')
print(f'  H0         : mean {TARGET_COL} = {BENCHMARK}')
print(f'  H1         : mean {TARGET_COL} != {BENCHMARK}  (two-tailed)')
print(f'  n          : {len(sample):,}')
print(f'  Sample mean: {sample.mean():.4f}')
print(f'  t-statistic: {t_stat:.4f}')
print(f'  p-value    : {p_val:.6f}')
conclusion = ('REJECT H0' if p_val < ALPHA else 'FAIL TO REJECT H0')
detail = (f'mean {TARGET_COL} is significantly different from {BENCHMARK}'
          if p_val < ALPHA
          else f'no significant difference from {BENCHMARK} detected')
print(f'  Conclusion : {conclusion} — {detail}  (p={p_val:.4f})')

In [ ]:
# ── Two-sample test: compare TARGET_COL across the two largest groups ─────────
GROUP_COL = COUNTRY_COL or next((c for c in CAT_COLS if 2 <= df_clean[c].nunique() <= 50), None)

if GROUP_COL:
    top2 = df_clean[GROUP_COL].value_counts().index[:2].tolist()
    g1   = df_clean[df_clean[GROUP_COL] == top2[0]][TARGET_COL].dropna()
    g2   = df_clean[df_clean[GROUP_COL] == top2[1]][TARGET_COL].dropna()

    use_param  = TARGET_COL in NORMAL_COLS
    test_label = 'Welch t-test' if use_param else 'Mann-Whitney U'

    if use_param:
        stat2, p2 = stats.ttest_ind(g1, g2, equal_var=False)
    else:
        stat2, p2 = stats.mannwhitneyu(g1, g2, alternative='two-sided')

    print(f'TWO-SAMPLE TEST  ({test_label})')
    print(f'  Comparing  : {TARGET_COL}')
    print(f'  H0         : distributions of {TARGET_COL} are equal across groups')
    print(f'  H1         : distributions differ')
    print(f'  Group 1 ({top2[0]}): n={len(g1):,}  median={g1.median():.2f}  mean={g1.mean():.2f}')
    print(f'  Group 2 ({top2[1]}): n={len(g2):,}  median={g2.median():.2f}  mean={g2.mean():.2f}')
    print(f'  Statistic  : {stat2:.4f}')
    print(f'  p-value    : {p2:.6f}')
    if p2 < ALPHA:
        print(f'  Conclusion : REJECT H0 — {TARGET_COL} differs significantly between groups'
              f'  (p={p2:.4f} < {ALPHA})')
    else:
        print(f'  Conclusion : FAIL TO REJECT H0 — no significant difference  (p={p2:.4f})')
else:
    print('No suitable grouping column found.')
    top2 = []; g1 = g2 = pd.Series(dtype=float)

In [ ]:
if GROUP_COL and len(g1) and len(g2):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for ax, grp, name in zip(axes, [g1, g2], top2):
        clipped = grp.clip(grp.quantile(0.01), grp.quantile(0.99))
        ax.hist(clipped, bins=50, color='steelblue', edgecolor='white', alpha=0.75)
        ax.axvline(grp.median(), color='tomato', lw=2, linestyle='--',
                   label=f'Median = {grp.median():.2f}')
        ax.set_title(f'{name}  (n={len(grp):,})')
        ax.set_xlabel(TARGET_COL)
        ax.set_ylabel('Count')
        ax.legend(fontsize=9)
    plt.suptitle(f'{TARGET_COL} by group  (p={p2:.4f})', fontsize=13)
    plt.tight_layout()
    plt.show()

---
## 5. Hypothesis Testing — Proportions & Frequencies (Chi-Square)

Chi-square tests work on **counts** (frequencies), not on means:

- **Goodness of Fit**: does a single categorical column follow an expected
  distribution (e.g., uniform)?
- **Test of Independence**: are two categorical columns statistically related?
  In e-commerce: *'Does the country a customer is from affect which type of
  product they purchase?'*

**Assumption**: all expected cell counts ≥ 5. We flag violations.
Standardised residuals (|res| > 2) reveal which cells drive the association.

In [ ]:
# ── Chi-square Goodness of Fit ───────────────────────────────────────────────
gof_col = COUNTRY_COL or next((c for c in CAT_COLS if 2 <= df_clean[c].nunique() <= 30), None)

if gof_col:
    observed = df_clean[gof_col].value_counts().head(10)
    expected = np.full(len(observed), observed.sum() / len(observed))

    chi2_gof, p_gof = stats.chisquare(f_obs=observed.values, f_exp=expected)
    dof_gof = len(observed) - 1

    print('CHI-SQUARE GOODNESS OF FIT')
    print(f'  Column     : {gof_col}  (top 10 categories)')
    print(f'  H0         : Categories are uniformly distributed')
    print(f'  H1         : Distribution is not uniform')
    print(f'  Chi2       : {chi2_gof:.4f}')
    print(f'  df         : {dof_gof}')
    print(f'  p-value    : {p_gof:.6f}')
    if p_gof < ALPHA:
        print(f'  Conclusion : REJECT H0 — distribution deviates from uniform  (p={p_gof:.4f})')
    else:
        print(f'  Conclusion : FAIL TO REJECT H0 — consistent with uniform  (p={p_gof:.4f})')

    fig, ax = plt.subplots(figsize=(11, 4))
    x = np.arange(len(observed))
    w = 0.35
    ax.bar(x - w/2, observed.values, w, label='Observed', color='steelblue', edgecolor='white')
    ax.bar(x + w/2, expected,        w, label='Expected (uniform)', color='salmon', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(observed.index, rotation=35, ha='right')
    ax.set_title(f'GoF Test — {gof_col}  (chi2={chi2_gof:.2f}, p={p_gof:.4f})')
    ax.set_ylabel('Count')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('No suitable column for GoF test.')
    gof_col = None

In [ ]:
# ── Chi-square Test of Independence ──────────────────────────────────────────
cat_low = [c for c in CAT_COLS if 2 <= df_clean[c].nunique() <= 30 and c != gof_col]
col_a = gof_col
col_b = cat_low[0] if cat_low else None

if col_a and col_b:
    top_a = df_clean[col_a].value_counts().head(6).index
    top_b = df_clean[col_b].value_counts().head(6).index
    sub_chi = df_clean[df_clean[col_a].isin(top_a) & df_clean[col_b].isin(top_b)]

    ct = pd.crosstab(sub_chi[col_a], sub_chi[col_b])
    chi2_ind, p_ind, dof_ind, expected_arr = stats.chi2_contingency(ct)
    low_exp = (expected_arr < 5).sum()

    print('CHI-SQUARE TEST OF INDEPENDENCE')
    print(f'  Columns    : {col_a}  x  {col_b}')
    print(f'  H0         : {col_a} and {col_b} are independent')
    print(f'  H1         : {col_a} and {col_b} are associated')
    print(f'  Chi2       : {chi2_ind:.4f}')
    print(f'  df         : {dof_ind}')
    print(f'  p-value    : {p_ind:.6f}')
    print(f'  Cells with expected < 5: {low_exp}  (assumption check)')
    if p_ind < ALPHA:
        print(f'  Conclusion : REJECT H0 — {col_a} and {col_b} are significantly associated')
    else:
        print(f'  Conclusion : FAIL TO REJECT H0 — no significant association')

    display(ct)

    # Standardised residuals heatmap
    std_resid = (ct.values - expected_arr) / np.sqrt(expected_arr)
    fig, ax = plt.subplots(figsize=(max(7, ct.shape[1] * 1.5), max(4, ct.shape[0] * 0.7)))
    sns.heatmap(pd.DataFrame(std_resid, index=ct.index, columns=ct.columns),
                annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax)
    ax.set_title(f'Standardised Residuals — {col_a} x {col_b}\n'
                 f'|residual| > 2 highlights cells driving the association')
    plt.tight_layout()
    plt.show()
else:
    print('Need at least two low-cardinality categorical columns.')
    col_a = col_b = None; chi2_ind = p_ind = 0

---
## 6. Hypothesis Testing — Multiple Groups (ANOVA / Kruskal-Wallis)

Comparing a numeric variable across **three or more groups** requires:

- **One-way ANOVA** (parametric): assumes normality and homoscedasticity of residuals.
- **Kruskal-Wallis** (non-parametric): rank-based alternative — used when normality is rejected.

A significant result only tells us *at least one* group differs.
**Tukey HSD post-hoc** identifies *which* pairs differ, controlling for
family-wise error rate.

In [ ]:
anova_cat = COUNTRY_COL or next((c for c in CAT_COLS if 3 <= df_clean[c].nunique() <= 30), None)
anova_num = TARGET_COL

if anova_cat:
    top_groups = df_clean[anova_cat].value_counts().head(6).index
    groups = [df_clean[df_clean[anova_cat] == g][anova_num].dropna().values
              for g in top_groups]
    groups = [g for g in groups if len(g) >= 5]

    use_param  = anova_num in NORMAL_COLS
    test_label = 'One-way ANOVA' if use_param else 'Kruskal-Wallis'

    stat_anova, p_anova = (stats.f_oneway(*groups) if use_param
                           else stats.kruskal(*groups))

    print(f'{test_label.upper()}')
    print(f'  Numeric col : {anova_num}')
    print(f'  Groups      : {list(top_groups)}')
    print(f'  H0          : All group distributions are equal')
    print(f'  H1          : At least one group differs')
    print(f'  Statistic   : {stat_anova:.4f}')
    print(f'  p-value     : {p_anova:.6f}')
    if p_anova < ALPHA:
        print(f'  Conclusion  : REJECT H0 — groups differ significantly  (p={p_anova:.4f})')
    else:
        print(f'  Conclusion  : FAIL TO REJECT H0  (p={p_anova:.4f})')
else:
    print('No suitable categorical column for ANOVA.')
    anova_cat = None; p_anova = 1.0; stat_anova = 0

In [ ]:
if anova_cat and p_anova < ALPHA:
    sub_anova = df_clean[df_clean[anova_cat].isin(top_groups)][[anova_cat, anova_num]].dropna()
    tukey = pairwise_tukeyhsd(endog=sub_anova[anova_num],
                               groups=sub_anova[anova_cat],
                               alpha=ALPHA)
    print('TUKEY HSD POST-HOC')
    print(tukey)

    tukey_df = pd.DataFrame(data=tukey._results_table.data[1:],
                             columns=tukey._results_table.data[0])
    sig_pairs = tukey_df[tukey_df['reject'] == True]
    print(f'\nSignificant pairs: {len(sig_pairs)} of {len(tukey_df)}')
    display(sig_pairs)
elif anova_cat:
    print('ANOVA/KW was not significant — post-hoc not warranted.')

In [ ]:
if anova_cat:
    sub_anova = df_clean[df_clean[anova_cat].isin(top_groups)][[anova_cat, anova_num]].dropna()
    clip_val  = sub_anova[anova_num].quantile(0.99)
    order_grp = (sub_anova.groupby(anova_cat)[anova_num]
                 .median().sort_values(ascending=False).index.tolist())

    fig, ax = plt.subplots(figsize=(12, 5))
    sns.boxplot(
        data=sub_anova.assign(**{anova_num: sub_anova[anova_num].clip(upper=clip_val)}),
        x=anova_cat, y=anova_num, order=order_grp, palette='muted', ax=ax,
        flierprops={'marker': 'o', 'alpha': 0.3, 'markersize': 3}
    )
    ax.set_title(f'{anova_num} by {anova_cat}  '
                 f'stat={stat_anova:.2f}, p={p_anova:.4f}  (clipped at 99th pct)')
    ax.set_xlabel(anova_cat)
    ax.set_ylabel(anova_num)
    ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()

---
## 7. Correlation Analysis

Correlation measures the **strength and direction** of a relationship between
two numeric variables, bounded in [−1, 1].

- **Pearson r**: measures linear association; appropriate when both variables
  are normally distributed.
- **Spearman ρ**: rank-based; robust to outliers and non-normality;
  captures monotonic (not just linear) relationships.

Correlation does **not** imply causation. In e-commerce terms, a strong
positive r between Quantity and Revenue confirms that larger orders reliably
generate more revenue — but the relationship's causal direction is obvious only
from domain knowledge.

In [ ]:
pearson_corr  = df_clean[NUMERIC_COLS].corr(method='pearson')
spearman_corr = df_clean[NUMERIC_COLS].corr(method='spearman')
mask = np.triu(np.ones_like(pearson_corr, dtype=bool))

n = len(NUMERIC_COLS)
fig, axes = plt.subplots(1, 2, figsize=(max(10, n * 2), max(5, n - 1)))
for ax, corr, title in zip(axes,
                             [pearson_corr, spearman_corr],
                             ['Pearson', 'Spearman']):
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, linewidths=0.5, vmin=-1, vmax=1, ax=ax)
    ax.set_title(f'{title} Correlation (lower triangle)')

plt.suptitle('Correlation Matrices', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Significance tests for all pairs
sig_rows = []
for i in range(len(NUMERIC_COLS)):
    for j in range(i + 1, len(NUMERIC_COLS)):
        c1, c2 = NUMERIC_COLS[i], NUMERIC_COLS[j]
        pair = df_clean[[c1, c2]].dropna()
        if len(pair) < 5:
            continue
        use_pearson = (c1 in NORMAL_COLS and c2 in NORMAL_COLS)
        r, p_corr = (stats.pearsonr(pair[c1], pair[c2])
                     if use_pearson
                     else stats.spearmanr(pair[c1], pair[c2]))
        sig_rows.append({
            'col_1': c1, 'col_2': c2,
            'method': 'Pearson' if use_pearson else 'Spearman',
            'r': round(r, 4), 'p_value': round(p_corr, 6),
            'significant': p_corr < ALPHA,
            'strength': ('strong' if abs(r) >= 0.7
                         else ('moderate' if abs(r) >= 0.4 else 'weak')),
        })

sig_df = pd.DataFrame(sig_rows).sort_values('r', key=abs, ascending=False)
display(sig_df)

In [ ]:
top_pairs = sig_df[sig_df['significant'] & (sig_df['r'].abs() >= 0.3)].head(4)

if top_pairs.empty:
    print('No significant pairs with |r| >= 0.3.')
else:
    fig, axes = plt.subplots(1, len(top_pairs), figsize=(5 * len(top_pairs), 5))
    if len(top_pairs) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, top_pairs.iterrows()):
        c1, c2 = row['col_1'], row['col_2']
        samp = df_clean[[c1, c2]].dropna().sample(min(3000, len(df_clean)), random_state=42)
        ax.scatter(samp[c1], samp[c2], alpha=0.3, s=10, color='steelblue')
        m, b = np.polyfit(samp[c1], samp[c2], 1)
        xr = np.linspace(samp[c1].min(), samp[c1].max(), 100)
        ax.plot(xr, m * xr + b, color='tomato', lw=2, label='OLS fit')
        ax.set_xlabel(c1); ax.set_ylabel(c2)
        ax.set_title(f'{c1} vs {c2}\n{row["method"]} r = {row["r"]:.3f}')
        ax.legend(fontsize=9)
    plt.suptitle('Scatter Plots — Top Correlated Pairs', fontsize=14)
    plt.tight_layout()
    plt.show()

---
## 8. Simple & Multiple Linear Regression

Linear regression models the expected value of a **target variable** as a
linear combination of predictor variables.

Key output statistics:
- **R²**: proportion of variance in the target explained by the model.
- **Adjusted R²**: penalises for adding unhelpful predictors — use for model comparison.
- **F-statistic p-value**: H0 is that all coefficients = 0 (i.e., model explains nothing).
- **Coefficient p-values**: whether each predictor contributes significantly.

Residual diagnostics check three OLS assumptions:
1. Residuals are random (no pattern vs fitted values → linearity)
2. Residuals are normally distributed (Q-Q plot)
3. Variance is constant (Scale-Location plot → homoscedasticity)

In [ ]:
TARGET = 'Revenue' if 'Revenue' in df_clean.columns else NUMERIC_COLS[-1]
preds_all = [c for c in NUMERIC_COLS if c != TARGET]

# Best single predictor: highest absolute correlation with TARGET
best_pred = (
    df_clean[preds_all + [TARGET]].corr()[TARGET]
    .drop(TARGET).abs().idxmax()
)

ols_data = df_clean[[best_pred, TARGET]].dropna()
X_s = sm.add_constant(ols_data[best_pred])
model_s = sm.OLS(ols_data[TARGET], X_s).fit()

print(f'SIMPLE OLS:  {TARGET}  ~  {best_pred}')
print(model_s.summary())

In [ ]:
# Top-3 correlated predictors
top_preds = (
    df_clean[preds_all + [TARGET]].corr()[TARGET]
    .drop(TARGET).abs()
    .sort_values(ascending=False)
    .head(3).index.tolist()
)

mlr_data = df_clean[top_preds + [TARGET]].dropna()
X_m = sm.add_constant(mlr_data[top_preds])
model_m = sm.OLS(mlr_data[TARGET], X_m).fit()

print(f'MULTIPLE OLS:  {TARGET}  ~  {" + ".join(top_preds)}')
print(model_m.summary())

In [ ]:
fitted    = model_m.fittedvalues
residuals = model_m.resid
std_resid = (residuals - residuals.mean()) / residuals.std()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Residuals vs Fitted
axes[0].scatter(fitted, residuals, alpha=0.3, s=10, color='steelblue')
axes[0].axhline(0, color='tomato', lw=1.5, linestyle='--')
axes[0].set_xlabel('Fitted values')
axes[0].set_ylabel('Residuals')
axes[0].set_title('Residuals vs Fitted\n(random scatter = linearity OK)')

# 2. Q-Q plot of residuals
stats.probplot(residuals, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot of Residuals\n(on line = normality OK)')

# 3. Scale-Location
axes[2].scatter(fitted, np.sqrt(np.abs(std_resid)), alpha=0.3, s=10, color='steelblue')
axes[2].set_xlabel('Fitted values')
axes[2].set_ylabel('sqrt(|Std Residuals|)')
axes[2].set_title('Scale-Location\n(flat line = homoscedasticity OK)')

plt.suptitle('Multiple Regression — Residual Diagnostics', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
coef = model_m.params.drop('const')
ci   = model_m.conf_int().drop('const')

fig, ax = plt.subplots(figsize=(9, max(3, len(coef) * 1.2)))
colors = ['tomato' if c < 0 else 'steelblue' for c in coef]
ax.barh(range(len(coef)), coef.values,
        xerr=[coef.values - ci[0].values, ci[1].values - coef.values],
        color=colors, capsize=5, edgecolor='white')
ax.set_yticks(range(len(coef)))
ax.set_yticklabels(coef.index)
ax.axvline(0, color='black', lw=1, linestyle='--')
ax.set_xlabel('Coefficient (with 95% CI)')
ax.set_title(f'MLR Coefficients — {TARGET}')
plt.tight_layout()
plt.show()

print(f'R\u00b2           : {model_m.rsquared:.4f}')
print(f'Adjusted R\u00b2  : {model_m.rsquared_adj:.4f}')
print(f'F-stat p-val : {model_m.f_pvalue:.6f}')
print(f'\nInterpretation: a 1-unit increase in each predictor is associated with '
      f'the listed coefficient change in {TARGET}, holding other predictors constant.')

---
## 9. Effect Size

A p-value tells us whether an effect **exists**; effect size tells us how
**large** or **practically meaningful** it is.  
With large samples (as here) even trivially small differences can become
statistically significant — always pair p-values with effect sizes.

| Test | Measure | Thresholds |
|------|---------|------------|
| t-test / MWU | Cohen's d | negligible < 0.2, small 0.2, medium 0.5, large 0.8 |
| Chi-square | Cramér's V | negligible < 0.1, small 0.1, medium 0.3, large 0.5 |
| ANOVA / KW | Eta-squared η² | negligible < 0.01, small 0.01, medium 0.06, large 0.14 |

In [ ]:
def cohen_d(a, b):
    na, nb = len(a), len(b)
    pooled = np.sqrt(((na-1)*a.std()**2 + (nb-1)*b.std()**2) / (na+nb-2))
    return (a.mean() - b.mean()) / pooled if pooled > 0 else float('nan')

def d_label(d):
    d = abs(d)
    return 'large' if d >= 0.8 else ('medium' if d >= 0.5 else ('small' if d >= 0.2 else 'negligible'))

d_val = None
if GROUP_COL and len(g1) and len(g2):
    d_val = cohen_d(g1, g2)
    print("EFFECT SIZE — Two-sample test  (Cohen's d)")
    print(f'  Groups     : {top2[0]}  vs  {top2[1]}')
    print(f"  Cohen's d  : {d_val:.4f}  [{d_label(d_val)}]")
    print(f'  Interpretation: group means are {abs(d_val):.2f} pooled SDs apart.')
else:
    print('Two-sample test groups not available.')

In [ ]:
def cramers_v(chi2_val, n, k, r):
    return np.sqrt(chi2_val / (n * (min(k, r) - 1)))

def v_label(v):
    return 'large' if v >= 0.5 else ('medium' if v >= 0.3 else ('small' if v >= 0.1 else 'negligible'))

V_val = None
if col_a and col_b:
    n_ct = ct.values.sum()
    V_val = cramers_v(chi2_ind, n_ct, ct.shape[1], ct.shape[0])
    print("EFFECT SIZE — Chi-square independence  (Cramér's V)")
    print(f'  Columns    : {col_a}  x  {col_b}')
    print(f"  Cramér's V : {V_val:.4f}  [{v_label(V_val)}]")
    print(f'  Interpretation: association strength is {v_label(V_val)}.')
else:
    print('Chi-square independence test not available.')

In [ ]:
def eta_squared(grp_list):
    all_v = np.concatenate(grp_list)
    gm    = all_v.mean()
    ss_b  = sum(len(g) * (g.mean() - gm)**2 for g in grp_list)
    ss_t  = ((all_v - gm)**2).sum()
    return ss_b / ss_t if ss_t > 0 else float('nan')

def eta_label(e):
    return 'large' if e >= 0.14 else ('medium' if e >= 0.06 else ('small' if e >= 0.01 else 'negligible'))

eta2_val = None
if anova_cat:
    eta2_val = eta_squared([pd.Series(g) for g in groups])
    print('EFFECT SIZE — ANOVA/Kruskal-Wallis  (Eta-squared)')
    print(f'  Numeric col : {anova_num}')
    print(f'  Eta-squared : {eta2_val:.4f}  [{eta_label(eta2_val)}]')
    print(f'  Interpretation: group membership explains {eta2_val*100:.1f}% of variance in {anova_num}.')
else:
    print('ANOVA not available.')

In [ ]:
summary_rows = []

if GROUP_COL and d_val is not None:
    summary_rows.append({
        'test': 'Two-sample (t / MWU)',
        'comparison': f'{top2[0]} vs {top2[1]}',
        'metric': TARGET_COL,
        'p_value': round(p2, 6),
        'effect_size': "Cohen's d",
        'value': round(d_val, 4),
        'magnitude': d_label(d_val),
    })

if col_a and col_b and V_val is not None:
    summary_rows.append({
        'test': 'Chi-square independence',
        'comparison': f'{col_a} x {col_b}',
        'metric': 'frequency',
        'p_value': round(p_ind, 6),
        'effect_size': "Cramér's V",
        'value': round(V_val, 4),
        'magnitude': v_label(V_val),
    })

if anova_cat and eta2_val is not None:
    summary_rows.append({
        'test': 'ANOVA / Kruskal-Wallis',
        'comparison': f'{anova_cat} groups',
        'metric': anova_num,
        'p_value': round(p_anova, 6),
        'effect_size': 'Eta-squared',
        'value': round(eta2_val, 4),
        'magnitude': eta_label(eta2_val),
    })

if summary_rows:
    display(pd.DataFrame(summary_rows))
else:
    print('No effect sizes to display.')

---
## 10. Key Takeaways

The cell below auto-generates a structured summary of all inferential findings,
tagging each as statistically significant, practically meaningful, or both.  
Assumption violations are flagged with recommended remedies.

In [ ]:
print('=' * 70)
print('  INFERENTIAL STATISTICS - KEY FINDINGS')
print('=' * 70)

# 1. Normality
n_norm  = len(NORMAL_COLS)
n_total = len(NUMERIC_COLS)
print(f'\n1. NORMALITY')
print(f'   {n_norm}/{n_total} numeric columns are normally distributed.')
print(f'   Non-normal columns were handled with non-parametric alternatives.')

# 2. Confidence intervals
print(f'\n2. CONFIDENCE INTERVALS  (95%)')
for col in biz_cols:
    r = ci_df.loc[col]
    print(f'   {col}: [{r["CI_lower_95"]:.2f},  {r["CI_upper_95"]:.2f}]  (mean={r["mean"]:.2f})')

# 3. Two-sample test
if GROUP_COL and len(top2) >= 2:
    sig = 'SIGNIFICANT' if p2 < ALPHA else 'not significant'
    eff = f"Cohen's d = {d_val:.3f} [{d_label(d_val)}]" if d_val is not None else ''
    print(f'\n3. TWO-SAMPLE TEST')
    print(f'   {TARGET_COL}: {top2[0]} vs {top2[1]} — {sig}  (p={p2:.4f})  {eff}')

# 4. Chi-square
if col_a and col_b:
    sig = 'SIGNIFICANT' if p_ind < ALPHA else 'not significant'
    eff = f"Cramér's V = {V_val:.3f} [{v_label(V_val)}]" if V_val is not None else ''
    print(f'\n4. CHI-SQUARE INDEPENDENCE')
    print(f'   {col_a} x {col_b} — {sig}  (p={p_ind:.4f})  {eff}')

# 5. ANOVA
if anova_cat:
    sig = 'SIGNIFICANT' if p_anova < ALPHA else 'not significant'
    eff = f'Eta-squared = {eta2_val:.3f} [{eta_label(eta2_val)}]' if eta2_val is not None else ''
    print(f'\n5. ANOVA / KRUSKAL-WALLIS')
    print(f'   {anova_num} across {anova_cat} — {sig}  (p={p_anova:.4f})  {eff}')

# 6. Correlation
strong_corr = sig_df[sig_df['strength'] == 'strong']
print(f'\n6. CORRELATION')
if len(strong_corr):
    for _, row in strong_corr.head(3).iterrows():
        print(f'   {row["col_1"]} ~ {row["col_2"]}: {row["method"]} r={row["r"]:.3f}  [{row["strength"]}]')
else:
    moderate = sig_df[(sig_df['strength'] == 'moderate') & sig_df['significant']]
    if len(moderate):
        for _, row in moderate.head(2).iterrows():
            print(f'   {row["col_1"]} ~ {row["col_2"]}: {row["method"]} r={row["r"]:.3f}  [{row["strength"]}]')
    else:
        print('   No strong or moderate significant correlations found.')

# 7. Regression
print(f'\n7. REGRESSION')
print(f'   Simple  OLS  ({best_pred} -> {TARGET}):  R\u00b2 = {model_s.rsquared:.3f}')
print(f'   Multiple OLS ({" + ".join(top_preds)} -> {TARGET}):')
print(f'     R\u00b2 = {model_m.rsquared:.3f},  Adj R\u00b2 = {model_m.rsquared_adj:.3f}')
sig_f = 'significant' if model_m.f_pvalue < ALPHA else 'NOT significant'
print(f'     F-test: {sig_f}  (p={model_m.f_pvalue:.4f})')

print('\n' + '-' * 70)
print('ASSUMPTION VIOLATIONS & REMEDIES:')
print('  - Most numeric columns are non-normal (typical for transactional data).')
print('    Remedy: non-parametric tests were used where required.')
print('  - Revenue is heavily right-skewed; OLS residuals are likely heteroscedastic.')
print('    Remedy: log-transform Revenue before regression to stabilise variance.')
print('  - Large sample size inflates statistical significance for small effects;')
print('    always interpret alongside effect sizes.')
print('=' * 70)